# 00. Стенд потоков данных и замера задержек

**Зачем.** Видеопотока, сырого IMU и сырых данных lidar/radar в ТЗ нет. Все детекторы угроз должны работать **только на полях потоков из разделов 5.2-5.3** и укладываться в требования по времени. Этот ноутбук делает общую основу для всех последующих:
1. **генератор потоков** с теми же полями, частотами, задержками и потерями, что в ТЗ (БВС-1, 2, 4, 5, 7; НСУ-2, 3);
2. **шину** сообщений в порядке *прибытия* (с джиттером и переупорядочиванием);
3. **стенд** для детекторов: время обработки каждого сообщения, задержка обнаружения, ложные тревоги, соответствие бюджету 200 мс.

## Что именно требует ТЗ (проверяется ниже)
| Поток | Частота | Допустимая задержка | Доставка |
|---|---|---|---|
| БВС-1 телеметрия | 5-10 Гц | < 500 мс | best-effort |
| БВС-2 ГНСС-метрики | 1-2 Гц | < 1 с | гарантированная |
| БВС-3 алерты (выход) | по событию | **< 200 мс (критично)** | гарантированная |
| БВС-4 препятствия | 5-20 Гц **при обнаружении, 0 Гц в штатном режиме** | < 300 мс | метаданные гарантированно |
| БВС-5 метео и энергия | 1 Гц | < 1 с | допустимы потери |
| БВС-7 диагностика | 0,1-0,2 Гц + по событию | < 2 с (алерты < 200 мс) | гарантированная для ошибок |
| НСУ-2 зоны | по событию + heartbeat 5 мин | < 1 с / < 5 с | гарантированная для критических |
| НСУ-3 метеопрогноз | 0,1-0,2 Гц, данные старше 30 с устарели | < 2 с | допустимы потери |

> **Допущения генератора (не из ТЗ):** физические связи погоды с полями (например, как фантомные возвраты lidar зависят от осадков), распределения задержек и потерь внутри допустимых границ. Это модель для проверки алгоритмов и времени, а не измеренные данные.

In [ ]:
import sys, pathlib, warnings, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.streams import (make_scenario, generate_bus, SPEC, NET, DT, TRUTH_KINDS)
from src.harness import (all_detectors, run_detectors, evaluate, Detector)
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200)

## 1. Скрытое состояние и потоки
Скрытый «истинный» процесс на сетке 10 Гц (высота, T, T_d, влажность, ветер, видимость, интенсивность осадков/пыли/дыма, лазер, запотевание линзы, ночная засветка) порождает значения полей потоков. Детекторы видят **только сообщения**: с частотами, задержками, потерями и шумом датчиков.

Сценарии: `clear`, `humid` (почти насыщенный воздух без осадков: ловушка для ложных тревог), `rain`, `snow`, `hail` (с дождём), `fog_cloud`, `dust`, `smoke`, `lens_fog` (снижение из сухого холодного слоя в тёплый влажный), `laser` (ночь над городом; зона может быть объявлена заранее в НСУ-2), `night_city`.

Ниже - обзор сценария (по полям, которые доступны модулю).

In [ ]:
def overview(kind, seed=3):
    cfg = make_scenario(kind, seed); msgs, tr, S = generate_bus(cfg)
    b1 = [(m["t_arrive"], m["p"]) for m in msgs if m["stream"] == "BVS5"]; t5 = np.array([a for a, _ in b1])
    fig, axs = plt.subplots(5, 1, figsize=(11, 11), sharex=True)
    axs[0].plot(S["t"], S["alt"], c="gray"); axs[0].set_ylabel("alt, м (БВС-1)")
    axs[1].plot(t5, [p["temperature"] for _, p in b1], label="T (БВС-5)"); axs[1].plot(S["t"], S["Td"], label="T_d (скрыта)", alpha=.7); axs[1].plot(S["t"], S["T_lens"], label="T линзы (скрыта)", alpha=.7)
    axs[1].legend(ncol=3, fontsize=8); axs[1].set_ylabel("°C")
    axs[2].plot(t5, [p["humidity"] for _, p in b1], c="C2"); axs[2].set_ylabel("RH, % (БВС-5)")
    axs[3].plot(t5, [p["wind_gust"] for _, p in b1], label="порывы", c="C3"); axs[3].plot(t5, [p["power_consumption"] / 50 for _, p in b1], label="мощность/50 Вт", c="C4"); axs[3].legend(fontsize=8)
    b4 = [m for m in msgs if m["stream"] == "BVS4"]; ys = {"lidar": 0, "radar": 1, "camera": 2}
    for ph, col in ((False, "C0"), (True, "C3")):
        pts = [(m["t_gen"], ys[m["p"]["sensor_type"]]) for m in b4 if m["p"]["_phantom"] == ph]
        if pts: axs[4].scatter(*zip(*pts), s=5, c=col, label="фантом (метка только для оценки)" if ph else "реальное препятствие")
    axs[4].set_yticks([0, 1, 2], ["lidar", "radar", "camera"]); axs[4].legend(fontsize=8, loc="upper right"); axs[4].set_ylabel("БВС-4: детекции")
    for a in axs:
        for k, v in tr.items():
            if v.any() and k != "night_city":
                on = S["t"][np.argmax(v)]; off = S["t"][len(v) - 1 - np.argmax(v[::-1])]; a.axvspan(on, off, alpha=.08, color={"rain": "b", "snow": "c", "hail": "m", "fog_cloud": "gray", "dust": "orange", "smoke": "k", "laser": "r", "lens_fog": "g"}[k])
        a.grid(alpha=.3)
    axs[4].set_xlabel("t, с"); axs[0].set_title(f"сценарий {kind}: метки истины - закрашенные интервалы ({', '.join(k for k, v in tr.items() if v.any())})"); plt.tight_layout(); plt.show()
overview("hail")

In [ ]:
cfg = make_scenario("clear", 3); msgs, tr, S = generate_bus(cfg)
fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
n3 = [m for m in msgs if m["stream"] == "NSU3"]
axs[0].semilogy([m["t_gen"] for m in n3], [m["p"]["visibility_m"] for m in n3], ".", label="НСУ-3 visibility_m (прогноз)"); axs[0].semilogy(S["t"], S["V"], label="истинная видимость (скрыта)"); axs[0].legend(); axs[0].grid(alpha=.3)
cfg = make_scenario("fog_cloud", 3); msgs, tr, S = generate_bus(cfg); n3 = [m for m in msgs if m["stream"] == "NSU3"]
axs[1].semilogy([m["t_gen"] for m in n3], [m["p"]["visibility_m"] for m in n3], ".", label="прогноз (fog_cloud)"); axs[1].semilogy(S["t"], S["V"], label="истинная"); axs[1].legend(); axs[1].grid(alpha=.3); axs[1].set_xlabel("t, с")
plt.tight_layout(); plt.show()

Внешний прогноз видимости (НСУ-3) **сглаживает** локальные события (в `fog_cloud` он сильно завышает видимость): на него нельзя опираться как на единственный источник.

## 2. Проверка генератора по ТЗ
Для каждого потока: фактическая частота, оценка потерь относительно номинального числа сообщений, перцентили задержки доставки (`t_arrive − t_gen`) против допустимой задержки из ТЗ, номинальный размер и пропускная способность. БВС-4: доля времени без сообщений в штатном режиме (препятствий нет).

In [ ]:
def stream_table(kind="clear", seed=5, duration=1200.0):
    cfg = make_scenario(kind, seed, duration=duration); cfg.obstacle_rate = 1e-9      # без препятствий: штатный режим БВС-4
    msgs, tr, S = generate_bus(cfg)
    df = pd.DataFrame([(m["stream"], m["t_gen"], m["t_arrive"], m["size"]) for m in msgs], columns=["s", "tg", "ta", "size"])
    nominal = {"BVS1": duration / 0.125, "BVS2": duration / 0.75, "BVS5": duration / 1.0, "BVS7": duration / 7.0, "NSU3": duration / 6.0}
    rows = []
    for s, g in df.groupby("s"):
        lat = (g.ta - g.tg).values; rate = len(g) / duration
        sp = SPEC[s]; rr = sp["rate"]
        ok_rate = (rate < 0.1) if s == "BVS4" else (rr[0] * 0.8 <= rate <= rr[1] * 1.05) if isinstance(rr, tuple) else True   # БВС-4: по событию, в штатном режиме ~0 Гц
        rows.append(dict(поток=s, частота_Гц=round(rate, 3), потери=round(1 - len(g) / nominal[s], 3) if s in nominal else np.nan,
                         lat_p50_мс=round(np.percentile(lat, 50) * 1e3), lat_p99_мс=round(np.percentile(lat, 99) * 1e3), допустимо_мс=int(sp["max_latency"] * 1e3),
                         p99_в_норме=bool(np.percentile(lat, 99) <= sp["max_latency"]), частота_в_диапазоне=ok_rate, кбит_с=round(len(g) / duration * sp["size"] * 8 / 1e3, 1)))
    return pd.DataFrame(rows).set_index("поток"), df
vt, df_clear = stream_table(); display(vt)
b1 = df_clear[df_clear.s == "BVS1"].sort_values("tg"); reorder = (b1.ta.diff() < 0).mean()
print(f"БВС-1: доля сообщений, пришедших раньше предыдущего по времени генерации (переупорядочивание): {reorder:.1%}")
print(f"БВС-4 в штатном режиме без препятствий: {(df_clear.s == 'BVS4').sum()} сообщений за 1200 с ({(df_clear.s == 'BVS4').sum() / 1200:.3f} Гц)")

Переупорядочивание реально: детекторы **обязаны использовать `timestamp`/`t_gen` для динамики**, а не время прибытия, и отбрасывать устаревшие сообщения (как делает эталонный детектор конденсата).

## 3. Слепое пятно БВС-4: поток молчит, когда сенсор «слепнет»
В штатном режиме БВС-4 молчит, поэтому отсутствие сообщений не отличить от «препятствий нет». Проверим, что происходит с реальными препятствиями при плохой видимости: **дистанция первого обнаружения** (по реальным, а не фантомным детекциям) для каждого сенсора.

In [ ]:
rows = []
for V0 in (100, 200, 300, 600, 1000, 3000, 30000):
    for sd in range(6):
        cfg = make_scenario("clear", 100 + sd); cfg.V0 = V0; cfg.obstacle_rate = 1 / 20; cfg.duration = 400.0
        msgs, tr, S = generate_bus(cfg); n_obs_est = 0
        for s in ("lidar", "radar", "camera"):
            ts = sorted((m["t_gen"], m["p"]["distance"]) for m in msgs if m["stream"] == "BVS4" and not m["p"]["_phantom"] and m["p"]["sensor_type"] == s)
            first = []; last_t = -99
            for t, d in ts:
                if t - last_t > 2.0: first.append(d)
                last_t = t
            rows.append(dict(V0=V0, sensor=s, first_detection_m=float(np.median(first)) if first else np.nan, n_approaches=len(first)))
sil = pd.DataFrame(rows).groupby(["V0", "sensor"]).agg(first_detection_m=("first_detection_m", "median"), n_approaches=("n_approaches", "sum")).unstack("sensor")
display(sil.round(0))
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for s in ("lidar", "radar", "camera"): ax.semilogx(sil.index, sil[("first_detection_m", s)], marker="o", label=s)
ax.set_xlabel("видимость V0, м"); ax.set_ylabel("дистанция первого обнаружения, м"); ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

При плотном тумане или пыли (V порядка сотен метров) lidar и камера теряют дальность, а **сообщений об этом нет**. Следствие для плана: потерю обзора можно выявлять только косвенно (сокращение дистанции первого обнаружения относительно radar, падение `detection_confidence`, `sensor_health_status` в тех сообщениях, что всё же приходят) и по метео-полям.

## 4. Стенд и эталонные быстрые детекторы
Эталонные детекторы нужны, чтобы проверить сам стенд и задать масштаб времени. Они простые, O(1) на сообщение:

| Детектор | Потоки | Идея |
|---|---|---|
| `condensation` | БВС-5 | Magnus + инерционное звено линзы + Φ((D+m)/σ) + гистерезис (из ноутбука 1) |
| `phantom_clutter` | БВС-4 (+БВС-5 как «тик») | частота «близких малых слабых» lidar-детекций в окне 10 с |
| `precip_rule` | БВС-5 | RH → вероятность осадков, тип по T_w (из ноутбука 2) |
| `hail_events` | БВС-7, БВС-1 | ≥ 3 события ошибки IMU или скачка pitch за 5 с |

Метрики: **recall** (тревога в окне [onset−60 с; конец+30 с]), **задержка** от начала события до *доставки* алерта в НСУ (путь: прибытие сообщения + обработка + канал БВС-3), **ложные тревоги в час** на сценариях без целевого события, **время обработки сообщения** (замер `perf_counter_ns`).

In [ ]:
kinds = ["clear", "humid", "rain", "snow", "hail", "fog_cloud", "dust", "smoke", "lens_fog", "laser", "night_city"]
t0 = time.time(); res, lat = evaluate(kinds, range(8)); print("время прогона стенда:", round(time.time() - t0, 1), "с")
pos = res[res.цель]; neg = res[~res.цель]
tab = pos.groupby(["детектор", "сценарий"]).agg(recall=("detected", "mean"), задержка_медиана_с=("delay", "median"), n=("seed", "count")).round(2)
display(tab)
display(neg.groupby("детектор").agg(ложных_в_час=("fa_per_hour", "mean"), сценариев=("seed", "count")).round(1))

Эти числа - **проверка стенда**, а не качество детекторов (параметры не тюнились, сценарии простые). Важно другое: стенд показывает, откуда берётся задержка. Для БВС-5 это в основном период 1 Гц и транспорт (p99 до 0,9 с), для окон по БВС-4 - длина окна, а не вычисления.

In [ ]:
lt = lat.groupby("детектор").proc_us.describe(percentiles=[.5, .9, .99, .999]).round(2)[["count", "mean", "50%", "90%", "99%", "99.9%", "max"]]
display(lt)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.boxplot([lat[lat.детектор == d].proc_us.clip(upper=50) for d in lt.index], showfliers=False); ax.set_xticks(range(1, len(lt) + 1), lt.index, fontsize=8)
ax.set_yscale("log"); ax.set_ylabel("время обработки сообщения, мкс (усы без выбросов)"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Бюджеты времени и запас по процессору
Требования ТЗ: алерт БВС-3 доставляется < 200 мс; сообщения БВС-4 < 300 мс. Время алерта `t_deliver = t_arrive + обработка + канал БВС-3`. Задержка транспорта *входящего* потока (до модуля) в бюджет 200 мс не входит и показана отдельно. Запас по процессору: загрузка = (сообщений в секунду) × (среднее время обработки) × (замедление бортового процессора относительно ноутбука).

In [ ]:
al = []
for kind in ("rain", "snow", "hail", "dust", "smoke", "lens_fog", "humid"):
    for sd in range(8):
        cfg = make_scenario(kind, sd); msgs, tr, S = generate_bus(cfg); dets = all_detectors(); alerts, _ = run_detectors(dets, msgs, np.random.default_rng(sd))
        for d, lst in alerts.items():
            for a in lst: al.append(dict(детектор=d, uplink=a["t_arrive"] - a["t_gen"], proc=a["proc_s"], down=a["down_s"]))
al = pd.DataFrame(al); al["путь_алерта_мс"] = (al.proc + al.down) * 1e3; al["с_uplink_мс"] = (al.uplink + al.proc + al.down) * 1e3
summ = al.groupby("детектор").agg(алертов=("proc", "size"), путь_алерта_p50=("путь_алерта_мс", "median"), путь_алерта_p99=("путь_алерта_мс", lambda x: np.percentile(x, 99)),
                                   доля_выше_200мс=("путь_алерта_мс", lambda x: (x > 200).mean()), uplink_p99_мс=("uplink", lambda x: np.percentile(x, 99) * 1e3)).round(2)
display(summ)
dur = 600.0; cfg = make_scenario("rain", 1); msgs, tr, S = generate_bus(cfg)
rates = pd.Series([m["stream"] for m in msgs]).value_counts() / cfg.duration
rows = []
for d in all_detectors():
    r_in = float(sum(rates.get(s, 0) for s in d.streams)); mean_us = lat[lat.детектор == d.name].proc_us.mean()
    rows.append(dict(детектор=d.name, сообщений_в_с=round(r_in, 1), среднее_мкс=round(mean_us, 2), загрузка_x1=r_in * mean_us * 1e-6, загрузка_x50=r_in * mean_us * 50e-6))
display(pd.DataFrame(rows).set_index("детектор").round(5))
fig, ax = plt.subplots(figsize=(7, 3.4)); ax.hist(al.путь_алерта_мс, bins=40); ax.axvline(200, c="r", ls="--", label="требование 200 мс"); ax.set_xlabel("путь алерта: обработка + канал БВС-3, мс"); ax.legend(); plt.tight_layout(); plt.show()

## 6. Как подключать новые детекторы
```python
class MyDetector(Detector):
    name = "my"; streams = ("BVS5", "BVS4"); targets = ("rain", "snow")   # метки истины, считающиеся «правильными»
    def reset(self): ...
    def on_message(self, m):            # m = {"stream","seq","t_gen","t_arrive","size","p": {поля из ТЗ}}
        return [dict(kind="...", ...)]  # алерты
```
Правила для всех детекторов: (1) только поля ТЗ (поле `_phantom` - метка истины для оценки, использовать нельзя); (2) динамика по `timestamp`/`t_gen`, устаревшие сообщения отбрасывать; (3) O(1) на сообщение, никаких блокирующих вызовов; (4) тяжёлые операции (облако точек по `point_cloud_ref`, HMM-сглаживание) - вне критического пути.

## 7. Выводы
**Что сделано.** Генератор потоков по схеме ТЗ (БВС-1, 2, 4, 5, 7; НСУ-2, 3), сетевая модель (задержки, потери, переупорядочивание), шина, стенд для детекторов с замером времени обработки, задержки обнаружения и ложных тревог, а также четыре эталонных O(1)-детектора для проверки самого стенда.

**Проверка генератора.** Частоты, потери и p99 задержек укладываются в допустимые значения ТЗ: БВС-1 ≈ 7,8 Гц (≈ 16 кбит/с, в ТЗ ≈ 20 кбит/с при 10 Гц), БВС-5 ≈ 0,97 Гц, НСУ-3 ≈ 0,16 Гц, БВС-4 в штатном режиме ≈ 0 Гц. **8 % сообщений БВС-1 приходят не в порядке генерации**: детекторы обязаны опираться на `timestamp`, а не на время прибытия.

**Слепое пятно БВС-4.** При видимости порядка сотен метров lidar и камера теряют дальность (при V = 100 м lidar не обнаруживает ни одного препятствия), а поток при этом молчит. Потерю обзора можно выявлять только косвенно: по сокращению дистанции первого обнаружения относительно radar, по `detection_confidence`, `sensor_health_status` и метеополям. Внешний прогноз `visibility_m` (НСУ-3) сглаживает локальные события и не заменяет бортовых признаков.

**Время.** Обработка сообщения эталонными детекторами занимает порядка 0,3-1,2 мкс (медиана), p99 ≤ 3,1 мкс, p99,9 ≈ 4-22 мкс, редкие выбросы до сотен мкс (сборщик мусора Python, прерывания). Загрузка процессора при номинальных частотах потоков пренебрежимо мала даже при замедлении в 50 раз (≤ 0,03 %). Путь алерта (обработка + канал БВС-3) укладывается в 200 мс с p99 ≈ 120-125 мс; около 0,3-1 % алертов превышают 200 мс **из-за хвоста транспорта**, а не вычислений. Это значит: жёсткие 200 мс алгоритмом не гарантируются, их определяет канал. Реальная задержка обнаружения определяется не вычислениями, а периодом потока (БВС-5: 1 Гц, p99 транспорта до 0,9 с; БВС-7: 0,14 Гц) и длиной окон усреднения (по БВС-4: ≈ 5-9 с).

**Эталонные детекторы (проверка стенда, не качество).** Конденсация по БВС-5: полнота 1,0, но 7,6 ложных тревог в час и «дребезг» (десятки алертов на сценарий): нужны минимальное время удержания и подавление повторов. Фантомы lidar в БВС-4: полнота 1,0 для дождя, снега, пыли, дыма и 0 ложных за 48 чистых сценариев. Правило по RH и T_w: полнота 0,75 (дождь), 0,62 (снег), 0 (град), 3,0 ложных в час. Детектор града по ошибкам IMU и скачкам pitch: полнота 1,0, задержка ≈ 0,6 с, 0 ложных.

**Ограничения.**
- Физические связи погоды с полями (например, частота фантомов lidar от осадков, зависимость дальностей от видимости) - наши допущения, не данные заказчика.
- Задержки и потери выбраны внутри допустимых границ ТЗ; реальные распределения могут отличаться.
- Время измерено на ноутбуке с Python; на бортовом процессоре и без интерпретатора оно будет иным (запас ×50 показан).
- Облако точек (`point_cloud_ref`) не моделируется: формат интенсивности в ТЗ не указан; тяжёлые алгоритмы по облаку вынесены из критического пути.

**Дальше:** ноутбук 1 (метео-классификатор и физика конденсата на БВС-5, с подавлением повторов), ноутбук 2 (фантомы БВС-4: расхождение lidar/radar/camera), ноутбук 3 (мощность и устойчивость: налипание, град), затем деградация камеры по `detection_confidence`, слияние видимости и интеграция.